# Feature Engineering

Convert raw OULAD interaction and assessment data into categorical student behavioural features for K-Modes clustering.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Project root: c:\Users\Abhijith Shetty\Downloads\E-Learning-Behavior-Segmentation
Raw data: c:\Users\Abhijith Shetty\Downloads\E-Learning-Behavior-Segmentation\data\raw
Processed data: c:\Users\Abhijith Shetty\Downloads\E-Learning-Behavior-Segmentation\data\processed


In [2]:
student_info = pd.read_csv(
    RAW_DIR / "studentInfo.csv"
)

vle = pd.read_csv(
    RAW_DIR / "vle.csv"
)

assessments = pd.read_csv(
    RAW_DIR / "assessments.csv"
)

student_assessment = pd.read_csv(
    RAW_DIR / "studentAssessment.csv"
)

courses = pd.read_csv(
    RAW_DIR / "courses.csv"
)

student_registration = pd.read_csv(
    RAW_DIR / "studentRegistration.csv"
)

student_vle = pd.read_csv(
    RAW_DIR / "studentVle.csv",
    dtype={
        "code_module": "category",
        "code_presentation": "category",
        "id_student": "int32",
        "id_site": "int32",
        "date": "int16",
        "sum_click": "int32"
    }
)

print("All datasets loaded.")

All datasets loaded.


In [3]:
print("studentInfo:", student_info.shape)
print("VLE:", vle.shape)
print("assessments:", assessments.shape)
print("studentAssessment:", student_assessment.shape)
print("studentVle:", student_vle.shape)

studentInfo: (32593, 12)
VLE: (6364, 6)
assessments: (206, 6)
studentAssessment: (173912, 5)
studentVle: (10655280, 6)


In [4]:
KEYS = [
    "id_student",
    "code_module",
    "code_presentation"
]

In [5]:
student_vle["week"] = student_vle["date"] // 7

In [6]:
activity_features = (
    student_vle
    .groupby(KEYS, observed=True)
    .agg(
        total_clicks=("sum_click", "sum"),
        active_days=("date", "nunique"),
        active_weeks=("week", "nunique"),
        first_activity=("date", "min"),
        last_activity=("date", "max"),
        unique_resources=("id_site", "nunique")
    )
    .reset_index()
)

activity_features["activity_span"] = (
    activity_features["last_activity"]
    - activity_features["first_activity"]
)

activity_features.head()

,id_student,code_module,code_presentation,total_clicks,active_days,active_weeks,first_activity,last_activity,unique_resources,activity_span
0,6516,AAA,2014J,2791,159,39,-23,269,84,292
1,8462,DDD,2013J,646,56,17,-6,118,125,124
2,8462,DDD,2014J,10,1,1,10,10,3,0
3,11391,AAA,2013J,934,40,27,-5,253,55,258
4,23629,BBB,2013B,161,16,11,-6,87,11,93


In [7]:
activity_features.shape

(29228, 10)

In [8]:
activity_features[
    [
        "total_clicks",
        "active_days",
        "active_weeks",
        "unique_resources",
        "activity_span"
    ]
].describe()

,total_clicks,active_days,active_weeks,unique_resources,activity_span
count,29228.000000,29228.000000,29228.000000,29228.000000,29228.000000
mean,1355.039654,61.862563,21.453093,67.075955,185.518578
std,1733.546045,54.027412,12.458364,54.953382,92.447016
min,1.000000,1.000000,1.000000,1.000000,0.000000
25%,260.750000,18.000000,10.000000,27.000000,104.000000
50%,739.500000,47.000000,23.000000,51.000000,234.000000
75%,1770.000000,92.000000,33.000000,93.000000,256.000000
max,24139.000000,286.000000,43.000000,413.000000,293.000000


In [9]:
def add_relative_category(
    df,
    value_column,
    new_column,
    labels
):
    percentiles = (
        df
        .groupby(
            ["code_module", "code_presentation"],
            observed=True
        )[value_column]
        .rank(method="average", pct=True)
    )

    df[new_column] = pd.cut(
        percentiles,
        bins=[0, 1/3, 2/3, 1],
        labels=labels,
        include_lowest=True
    )

    return df

In [10]:
activity_features = add_relative_category(
    activity_features,
    "total_clicks",
    "Activity_Level",
    ["Low", "Medium", "High"]
)

In [11]:
activity_features["Activity_Level"].value_counts()

Activity_Level
High      9749
Medium    9743
Low       9736
Name: count, dtype: int64

In [12]:
activity_features = add_relative_category(
    activity_features,
    "active_days",
    "Learning_Frequency",
    ["Rare", "Moderate", "Frequent"]
)

In [13]:
activity_features["Learning_Frequency"].value_counts()

Learning_Frequency
Moderate    9765
Frequent    9746
Rare        9717
Name: count, dtype: int64

## Improved Learning Consistency and Inactivity Features

In [14]:
enrollment_window = (
    activity_features[KEYS]
    .merge(
        courses[
            [
                "code_module",
                "code_presentation",
                "module_presentation_length"
            ]
        ],
        on=[
            "code_module",
            "code_presentation"
        ],
        how="left"
    )
    .merge(
        student_registration[
            KEYS + ["date_unregistration"]
        ],
        on=KEYS,
        how="left"
    )
)

enrollment_window.head()

,id_student,code_module,code_presentation,module_presentation_length,date_unregistration
0,6516,AAA,2014J,269,NaN
1,8462,DDD,2013J,261,119.0
2,8462,DDD,2014J,262,18.0
3,11391,AAA,2013J,268,NaN
4,23629,BBB,2013B,240,NaN


In [15]:
enrollment_window["effective_end"] = (
    enrollment_window["module_presentation_length"]
)

withdrawn = (
    enrollment_window["date_unregistration"].notna()
    & (enrollment_window["date_unregistration"] >= 0)
)

enrollment_window.loc[
    withdrawn,
    "effective_end"
] = np.minimum(
    enrollment_window.loc[
        withdrawn,
        "date_unregistration"
    ],
    enrollment_window.loc[
        withdrawn,
        "module_presentation_length"
    ]
)

enrollment_window["effective_end"] = (
    enrollment_window["effective_end"]
    .clip(lower=0)
)

enrollment_window.head()

,id_student,code_module,code_presentation,module_presentation_length,date_unregistration,effective_end
0,6516,AAA,2014J,269,NaN,269
1,8462,DDD,2013J,261,119.0,119
2,8462,DDD,2014J,262,18.0,18
3,11391,AAA,2013J,268,NaN,268
4,23629,BBB,2013B,240,NaN,240


In [16]:
course_activity = (
    student_vle.loc[
        student_vle["date"] >= 0,
        KEYS + ["date"]
    ]
    .drop_duplicates()
)

In [17]:
course_activity = course_activity.merge(
    enrollment_window[
        KEYS + ["effective_end"]
    ],
    on=KEYS,
    how="inner"
)

course_activity = course_activity[
    course_activity["date"]
    <= course_activity["effective_end"]
].copy()

course_activity["course_week"] = (
    course_activity["date"] // 7
)

course_activity.head()

,id_student,code_module,code_presentation,date,effective_end,course_week
0,345357,AAA,2013J,0,268,0
1,336207,AAA,2013J,0,268,0
2,331358,AAA,2013J,0,53,0
3,321942,AAA,2013J,0,268,0
4,324002,AAA,2013J,0,268,0


In [18]:
temporal_features = (
    course_activity
    .groupby(KEYS, observed=True)
    .agg(
        active_weeks_course=("course_week", "nunique"),
        first_course_activity=("date", "min"),
        last_course_activity=("date", "max")
    )
    .reset_index()
)

In [19]:
temporal_features = enrollment_window[
    KEYS + ["effective_end"]
].merge(
    temporal_features,
    on=KEYS,
    how="left"
)

temporal_features["active_weeks_course"] = (
    temporal_features["active_weeks_course"]
    .fillna(0)
)

In [20]:
temporal_features["total_course_weeks"] = np.ceil(
    (
        temporal_features["effective_end"] + 1
    ) / 7
).clip(lower=1)

In [21]:
temporal_features["consistency_ratio"] = (
    temporal_features["active_weeks_course"]
    / temporal_features["total_course_weeks"]
).clip(0, 1)

In [22]:
temporal_features["Learning_Consistency"] = pd.cut(
    temporal_features["consistency_ratio"],
    bins=[
        -0.001,
        1/3,
        2/3,
        1.0
    ],
    labels=[
        "Irregular",
        "Moderately Regular",
        "Consistent"
    ],
    include_lowest=True
)

In [23]:
temporal_features["Learning_Consistency"].value_counts()

Learning_Consistency
Consistent            15026
Moderately Regular     8050
Irregular              6152
Name: count, dtype: int64

In [24]:
course_activity = course_activity.sort_values(
    KEYS + ["date"]
)

In [25]:
course_activity["internal_gap"] = (
    course_activity
    .groupby(KEYS, observed=True)["date"]
    .diff()
    .sub(1)
    .clip(lower=0)
)

In [26]:
internal_gaps = (
    course_activity
    .groupby(KEYS, observed=True)["internal_gap"]
    .max()
    .reset_index(name="max_internal_gap")
)

In [27]:
temporal_features = temporal_features.merge(
    internal_gaps,
    on=KEYS,
    how="left"
)

temporal_features["max_internal_gap"] = (
    temporal_features["max_internal_gap"]
    .fillna(0)
)

In [28]:
temporal_features["start_gap"] = (
    temporal_features["first_course_activity"]
)

temporal_features["end_gap"] = (
    temporal_features["effective_end"]
    - temporal_features["last_course_activity"]
)

In [29]:
no_course_activity = (
    temporal_features["first_course_activity"].isna()
)

temporal_features.loc[
    no_course_activity,
    "start_gap"
] = temporal_features.loc[
    no_course_activity,
    "effective_end"
]

temporal_features.loc[
    no_course_activity,
    "end_gap"
] = 0

In [30]:
temporal_features[
    ["start_gap", "end_gap"]
] = temporal_features[
    ["start_gap", "end_gap"]
].fillna(0)

In [31]:
temporal_features["max_inactive_days"] = (
    temporal_features[
        [
            "start_gap",
            "max_internal_gap",
            "end_gap"
        ]
    ]
    .max(axis=1)
)

In [32]:
temporal_features[
    [
        "active_weeks_course",
        "total_course_weeks",
        "consistency_ratio",
        "Learning_Consistency",
        "max_inactive_days"
    ]
].head(20)

,active_weeks_course,total_course_weeks,consistency_ratio,Learning_Consistency,max_inactive_days
0,35.0,39.0,0.897436,Consistent,26.0
1,16.0,18.0,0.888889,Consistent,10.0
2,1.0,3.0,0.333333,Irregular,10.0
3,26.0,39.0,0.666667,Moderately Regular,23.0
4,10.0,35.0,0.285714,Irregular,153.0
5,24.0,39.0,0.615385,Moderately Regular,38.0
6,33.0,39.0,0.846154,Consistent,18.0
7,17.0,35.0,0.485714,Moderately Regular,39.0
8,33.0,35.0,0.942857,Consistent,8.0
9,31.0,38.0,0.815789,Consistent,21.0


In [33]:
activity_features = activity_features.drop(
    columns=[
        "Learning_Consistency",
        "consistency_ratio",
        "Inactivity_Pattern",
        "median_activity_gap",
        "max_activity_gap",
        "possible_active_weeks"
    ],
    errors="ignore"
)

In [34]:
activity_features = activity_features.merge(
    temporal_features[
        KEYS + [
            "active_weeks_course",
            "total_course_weeks",
            "consistency_ratio",
            "Learning_Consistency",
            "max_inactive_days"
        ]
    ],
    on=KEYS,
    how="left"
)

In [35]:
activity_features = add_relative_category(
    activity_features,
    "max_inactive_days",
    "Inactivity_Pattern",
    [
        "Low Inactivity",
        "Moderate Inactivity",
        "High Inactivity"
    ]
)

In [36]:
print(
    activity_features["Learning_Consistency"]
    .value_counts()
)

print(
    activity_features["Inactivity_Pattern"]
    .value_counts()
)

Learning_Consistency
Consistent            15026
Moderately Regular     8050
Irregular              6152
Name: count, dtype: int64
Inactivity_Pattern
Moderate Inactivity    9813
High Inactivity        9745
Low Inactivity         9670
Name: count, dtype: int64


In [37]:
activity_features["Learning_Consistency"].value_counts()

Learning_Consistency
Consistent            15026
Moderately Regular     8050
Irregular              6152
Name: count, dtype: int64

In [38]:
activity_features["Inactivity_Pattern"].value_counts()

Inactivity_Pattern
Moderate Inactivity    9813
High Inactivity        9745
Low Inactivity         9670
Name: count, dtype: int64

In [39]:
print("Learning Consistency:")
print(
    activity_features["Learning_Consistency"]
    .value_counts(dropna=False)
)

print("\nInactivity Pattern:")
print(
    activity_features["Inactivity_Pattern"]
    .value_counts(dropna=False)
)

Learning Consistency:
Learning_Consistency
Consistent            15026
Moderately Regular     8050
Irregular              6152
Name: count, dtype: int64

Inactivity Pattern:
Inactivity_Pattern
Moderate Inactivity    9813
High Inactivity        9745
Low Inactivity         9670
Name: count, dtype: int64


In [40]:
activity_features[
    [
        "Learning_Consistency",
        "Inactivity_Pattern"
    ]
].isnull().sum()

Learning_Consistency    0
Inactivity_Pattern      0
dtype: int64

activity_features["possible_active_weeks"] = (
    activity_features["activity_span"] // 7
) + 1

activity_features["consistency_ratio"] = (
    activity_features["active_weeks"]
    / activity_features["possible_active_weeks"]
)

activity_features["consistency_ratio"] = (
    activity_features["consistency_ratio"]
    .clip(0, 1)
)

activity_features["Learning_Consistency"] = pd.cut(
    activity_features["consistency_ratio"],
    bins=[-0.01, 0.4, 0.7, 1.0],
    labels=[
        "Irregular",
        "Moderately Regular",
        "Consistent"
    ]
)

activity_features["Learning_Consistency"].value_counts()

active_dates = (
    student_vle[
        KEYS + ["date"]
    ]
    .drop_duplicates()
    .sort_values(KEYS + ["date"])
)

active_dates["activity_gap"] = (
    active_dates
    .groupby(KEYS, observed=True)["date"]
    .diff()
)

gap_features = (
    active_dates
    .groupby(KEYS, observed=True)
    .agg(
        median_activity_gap=("activity_gap", "median"),
        max_activity_gap=("activity_gap", "max")
    )
    .reset_index()
)

gap_features.head()

activity_features = activity_features.merge(
    gap_features,
    on=KEYS,
    how="left"
)

activity_features[
    ["median_activity_gap", "max_activity_gap"]
] = activity_features[
    ["median_activity_gap", "max_activity_gap"]
].fillna(0)

activity_features = add_relative_category(
    activity_features,
    "max_activity_gap",
    "Inactivity_Pattern",
    [
        "Low Inactivity",
        "Moderate Inactivity",
        "High Inactivity"
    ]
)

activity_features["Inactivity_Pattern"].value_counts()

## Content Preference

In [41]:
vle_lookup = vle[
    [
        "code_module",
        "code_presentation",
        "id_site",
        "activity_type"
    ]
].copy()

In [42]:
activity_with_type = student_vle[
    KEYS + [
        "id_site",
        "sum_click"
    ]
].merge(
    vle_lookup,
    on=[
        "code_module",
        "code_presentation",
        "id_site"
    ],
    how="left"
)

In [43]:
activity_with_type.head()

,id_student,code_module,code_presentation,id_site,sum_click,activity_type
0,28400,AAA,2013J,546652,4,forumng
1,28400,AAA,2013J,546652,1,forumng
2,28400,AAA,2013J,546652,1,forumng
3,28400,AAA,2013J,546614,11,homepage
4,28400,AAA,2013J,546714,1,oucontent


In [44]:
activity_with_type["activity_type"].value_counts()

activity_type
forumng           2408457
oucontent         1963782
subpage           1949898
homepage          1735226
quiz               914573
resource           813185
url                407716
ouwiki             178165
oucollaborate       65793
externalquiz        50910
page                47549
questionnaire       44905
ouelluminate        22360
glossary            17435
dataplus            13506
dualpane            10153
htmlactivity         6835
folder               4678
sharedsubpage         148
repeatactivity          6
Name: count, dtype: int64

In [45]:
activity_with_type["activity_type"].isna().sum()

np.int64(0)

In [46]:
def map_activity_category(activity_type):

    learning_content = {
        "oucontent",
        "resource",
        "page",
        "subpage",
        "sharedsubpage",
        "folder",
        "glossary",
        "htmlactivity",
        "dualpane",
        "repeatactivity",
        "dataplus"
    }

    discussion = {
        "forumng",
        "ouwiki",
        "oucollaborate",
        "ouelluminate"
    }

    assessment = {
        "quiz",
        "questionnaire",
        "externalquiz"
    }

    navigation = {
        "homepage"
    }

    external = {
        "url"
    }

    if activity_type in learning_content:
        return "Learning Content"

    if activity_type in discussion:
        return "Discussion"

    if activity_type in assessment:
        return "Assessment"

    if activity_type in navigation:
        return "Navigation"

    if activity_type in external:
        return "External Resource"

    return "Other"

In [47]:
activity_with_type["content_category"] = (
    activity_with_type["activity_type"]
    .apply(map_activity_category)
)

In [48]:
activity_with_type["content_category"].value_counts()

content_category
Learning Content     4827175
Discussion           2674775
Navigation           1735226
Assessment           1010388
External Resource     407716
Name: count, dtype: int64

In [49]:
activity_with_type.loc[
    activity_with_type["content_category"] == "Other",
    "activity_type"
].value_counts()

Series([], Name: count, dtype: int64)

In [50]:
content_clicks = (
    activity_with_type
    .groupby(
        KEYS + ["content_category"],
        observed=True
    )["sum_click"]
    .sum()
    .reset_index(name="category_clicks")
)

In [51]:
content_clicks.head()

,id_student,code_module,code_presentation,content_category,category_clicks
0,6516,AAA,2014J,Discussion,451
1,6516,AAA,2014J,External Resource,143
2,6516,AAA,2014J,Learning Content,1700
3,6516,AAA,2014J,Navigation,497
4,8462,DDD,2013J,Assessment,12


In [52]:
preferred_content = (
    content_clicks
    .sort_values(
        "category_clicks",
        ascending=False
    )
    .drop_duplicates(
        subset=KEYS
    )
)

In [53]:
preferred_content = preferred_content[
    KEYS + ["content_category"]
].rename(
    columns={
        "content_category": "Content_Preference"
    }
)

In [54]:
preferred_content["Content_Preference"].value_counts()

Content_Preference
Learning Content     16923
Discussion            5609
Assessment            4384
Navigation            2304
External Resource        8
Name: count, dtype: int64

In [55]:
# Merge the preferred content category safely.
# Dropping first makes this cell safe to rerun without creating _x/_y columns.
activity_features = activity_features.drop(
    columns=["Content_Preference"],
    errors="ignore"
)

activity_features = activity_features.merge(
    preferred_content,
    on=KEYS,
    how="left"
)

# Merge the extremely rare External Resource category into Learning Content.
activity_features["Content_Preference"] = (
    activity_features["Content_Preference"]
    .replace({
        "External Resource": "Learning Content"
    })
)

activity_features["Content_Preference"].value_counts(dropna=False)


Content_Preference
Learning Content    16931
Discussion           5609
Assessment           4384
Navigation           2304
Name: count, dtype: int64

In [56]:
resource_diversity = (
    activity_with_type
    .groupby(KEYS, observed=True)["activity_type"]
    .nunique()
    .reset_index(
        name="unique_activity_types"
    )
)

In [57]:
activity_features = activity_features.merge(
    resource_diversity,
    on=KEYS,
    how="left"
)

In [58]:
activity_features = add_relative_category(
    activity_features,
    "unique_activity_types",
    "Resource_Diversity",
    [
        "Narrow",
        "Moderate",
        "Diverse"
    ]
)

In [59]:
activity_features["Resource_Diversity"].value_counts()

Resource_Diversity
Diverse     10437
Moderate     9920
Narrow       8871
Name: count, dtype: int64

In [60]:
assessment_data = student_assessment.merge(
    assessments[
        [
            "id_assessment",
            "code_module",
            "code_presentation",
            "assessment_type",
            "date"
        ]
    ],
    on="id_assessment",
    how="left"
)

In [61]:
assessment_data.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0


In [62]:
total_assessments = (
    assessments
    .groupby(
        [
            "code_module",
            "code_presentation"
        ]
    )["id_assessment"]
    .nunique()
    .reset_index(
        name="total_assessments"
    )
)

In [63]:
assessment_attempts = (
    assessment_data
    .groupby(KEYS)["id_assessment"]
    .nunique()
    .reset_index(
        name="assessments_attempted"
    )
)

In [64]:
assessment_attempts = assessment_attempts.merge(
    total_assessments,
    on=[
        "code_module",
        "code_presentation"
    ],
    how="left"
)

In [65]:
assessment_attempts["assessment_attempt_ratio"] = (
    assessment_attempts["assessments_attempted"]
    / assessment_attempts["total_assessments"]
)

In [66]:
assessment_attempts.head()

,id_student,code_module,code_presentation,assessments_attempted,total_assessments,assessment_attempt_ratio
0,6516,AAA,2014J,5,6,0.833333
1,8462,DDD,2013J,3,7,0.428571
2,8462,DDD,2014J,4,7,0.571429
3,11391,AAA,2013J,5,6,0.833333
4,23629,BBB,2013B,4,12,0.333333


In [67]:
assessment_attempts["Assessment_Engagement"] = pd.cut(
    assessment_attempts["assessment_attempt_ratio"],
    bins=[-0.01, 0.5, 0.8, 1.0],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [68]:
assessment_attempts["Assessment_Engagement"].value_counts()

Assessment_Engagement
High      13497
Low        7697
Medium     4649
Name: count, dtype: int64

In [69]:
activity_features = activity_features.merge(
    assessment_attempts[
        KEYS + [
            "assessments_attempted",
            "total_assessments",
            "assessment_attempt_ratio",
            "Assessment_Engagement"
        ]
    ],
    on=KEYS,
    how="left"
)

In [70]:
activity_features["assessments_attempted"] = (
    activity_features["assessments_attempted"]
    .fillna(0)
)

activity_features["assessment_attempt_ratio"] = (
    activity_features["assessment_attempt_ratio"]
    .fillna(0)
)

activity_features["Assessment_Engagement"] = (
    activity_features["Assessment_Engagement"]
    .astype("object")
    .fillna("Low")
)

In [71]:
assessment_data["submission_difference"] = (
    assessment_data["date_submitted"]
    - assessment_data["date"]
)

In [72]:
dated_submissions = assessment_data[
    assessment_data["date"].notna()
].copy()

In [73]:
submission_features = (
    dated_submissions
    .groupby(KEYS)["submission_difference"]
    .median()
    .reset_index(
        name="median_submission_difference"
    )
)

In [74]:
def classify_submission(value):

    if pd.isna(value):
        return "No Submission Data"

    if value <= -3:
        return "Early"

    if value <= 0:
        return "On-Time"

    return "Late"

In [75]:
submission_features["Submission_Behaviour"] = (
    submission_features["median_submission_difference"]
    .apply(classify_submission)
)

In [76]:
submission_features["Submission_Behaviour"].value_counts()

Submission_Behaviour
On-Time    10215
Late        8491
Early       7136
Name: count, dtype: int64

In [77]:
activity_features = activity_features.merge(
    submission_features,
    on=KEYS,
    how="left"
)

In [78]:
activity_features["Submission_Behaviour"] = (
    activity_features["Submission_Behaviour"]
    .fillna("No Submission Data")
)

In [79]:
student_results = student_info[
    KEYS + ["final_result"]
].drop_duplicates()

In [80]:
student_behaviour = activity_features.merge(
    student_results,
    on=KEYS,
    how="left"
)

In [81]:
student_behaviour.head()

,id_student,code_module,code_presentation,total_clicks,active_days,active_weeks,first_activity,last_activity,unique_resources,activity_span,...,Content_Preference,unique_activity_types,Resource_Diversity,assessments_attempted,total_assessments,assessment_attempt_ratio,Assessment_Engagement,median_submission_difference,Submission_Behaviour,final_result
0,6516,AAA,2014J,2791,159,39,-23,269,84,292,...,Learning Content,7,Moderate,5.0,6.0,0.833333,High,-2.0,On-Time,Pass
1,8462,DDD,2013J,646,56,17,-6,118,125,124,...,Learning Content,9,Moderate,3.0,7.0,0.428571,Low,-2.0,On-Time,Withdrawn
2,8462,DDD,2014J,10,1,1,10,10,3,0,...,Navigation,3,Narrow,4.0,7.0,0.571429,Medium,-52.5,Early,Withdrawn
3,11391,AAA,2013J,934,40,27,-5,253,55,258,...,Learning Content,6,Narrow,5.0,6.0,0.833333,High,-2.0,On-Time,Pass
4,23629,BBB,2013B,161,16,11,-6,87,11,93,...,Discussion,5,Narrow,4.0,12.0,0.333333,Low,6.0,Late,Fail


In [82]:
BEHAVIOUR_FEATURES = [
    "Activity_Level",
    "Learning_Frequency",
    "Learning_Consistency",
    "Content_Preference",
    "Resource_Diversity",
    "Assessment_Engagement",
    "Submission_Behaviour",
    "Inactivity_Pattern"
]

In [83]:
student_behaviour[
    KEYS + BEHAVIOUR_FEATURES
].head(20)

,id_student,code_module,code_presentation,Activity_Level,Learning_Frequency,Learning_Consistency,Content_Preference,Resource_Diversity,Assessment_Engagement,Submission_Behaviour,Inactivity_Pattern
0,6516,AAA,2014J,High,Frequent,Consistent,Learning Content,Moderate,High,On-Time,Moderate Inactivity
1,8462,DDD,2013J,Medium,Moderate,Consistent,Learning Content,Moderate,Low,On-Time,Low Inactivity
2,8462,DDD,2014J,Low,Rare,Irregular,Navigation,Narrow,Medium,Early,Low Inactivity
3,11391,AAA,2013J,Medium,Rare,Moderately Regular,Learning Content,Narrow,High,On-Time,Moderate Inactivity
4,23629,BBB,2013B,Low,Rare,Irregular,Discussion,Narrow,Low,Late,High Inactivity
5,23698,CCC,2014J,Medium,Frequent,Moderately Regular,Assessment,Diverse,High,Late,High Inactivity
6,23798,BBB,2013J,Medium,Frequent,Consistent,Navigation,Diverse,High,On-Time,Low Inactivity
7,24186,GGG,2014B,Low,Moderate,Moderately Regular,Assessment,Moderate,Medium,Early,Moderate Inactivity
8,24213,DDD,2014B,High,Frequent,Consistent,Discussion,Moderate,High,Late,Low Inactivity
9,24391,GGG,2013J,High,Frequent,Consistent,Learning Content,Diverse,High,Early,Low Inactivity


In [84]:
student_behaviour[
    BEHAVIOUR_FEATURES
].isnull().sum()

Activity_Level           0
Learning_Frequency       0
Learning_Consistency     0
Content_Preference       0
Resource_Diversity       0
Assessment_Engagement    0
Submission_Behaviour     0
Inactivity_Pattern       0
dtype: int64

In [85]:
for feature in BEHAVIOUR_FEATURES:
    print("\n", feature)
    print(student_behaviour[feature].value_counts(dropna=False))


 Activity_Level
Activity_Level
High      9749
Medium    9743
Low       9736
Name: count, dtype: int64

 Learning_Frequency
Learning_Frequency
Moderate    9765
Frequent    9746
Rare        9717
Name: count, dtype: int64

 Learning_Consistency
Learning_Consistency
Consistent            15026
Moderately Regular     8050
Irregular              6152
Name: count, dtype: int64

 Content_Preference
Content_Preference
Learning Content    16931
Discussion           5609
Assessment           4384
Navigation           2304
Name: count, dtype: int64

 Resource_Diversity
Resource_Diversity
Diverse     10437
Moderate     9920
Narrow       8871
Name: count, dtype: int64

 Assessment_Engagement
Assessment_Engagement
High      13493
Low       11093
Medium     4642
Name: count, dtype: int64

 Submission_Behaviour
Submission_Behaviour
On-Time               10208
Late                   8486
Early                  7099
No Submission Data     3435
Name: count, dtype: int64

 Inactivity_Pattern
Inactivity_Pa

In [86]:
student_behaviour.to_csv(
    PROCESSED_DIR / "student_behaviour.csv",
    index=False
)

print(
    "Saved:",
    PROCESSED_DIR / "student_behaviour.csv"
)

Saved: c:\Users\Abhijith Shetty\Downloads\E-Learning-Behavior-Segmentation\data\processed\student_behaviour.csv


In [87]:
(PROCESSED_DIR / "student_behaviour.csv").exists()

True

In [88]:
student_behaviour.shape

(29228, 28)

In [89]:
student_behaviour[
    KEYS
    + BEHAVIOUR_FEATURES
    + ["final_result"]
].head(10)

,id_student,code_module,code_presentation,Activity_Level,Learning_Frequency,Learning_Consistency,Content_Preference,Resource_Diversity,Assessment_Engagement,Submission_Behaviour,Inactivity_Pattern,final_result
0,6516,AAA,2014J,High,Frequent,Consistent,Learning Content,Moderate,High,On-Time,Moderate Inactivity,Pass
1,8462,DDD,2013J,Medium,Moderate,Consistent,Learning Content,Moderate,Low,On-Time,Low Inactivity,Withdrawn
2,8462,DDD,2014J,Low,Rare,Irregular,Navigation,Narrow,Medium,Early,Low Inactivity,Withdrawn
3,11391,AAA,2013J,Medium,Rare,Moderately Regular,Learning Content,Narrow,High,On-Time,Moderate Inactivity,Pass
4,23629,BBB,2013B,Low,Rare,Irregular,Discussion,Narrow,Low,Late,High Inactivity,Fail
5,23698,CCC,2014J,Medium,Frequent,Moderately Regular,Assessment,Diverse,High,Late,High Inactivity,Pass
6,23798,BBB,2013J,Medium,Frequent,Consistent,Navigation,Diverse,High,On-Time,Low Inactivity,Distinction
7,24186,GGG,2014B,Low,Moderate,Moderately Regular,Assessment,Moderate,Medium,Early,Moderate Inactivity,Pass
8,24213,DDD,2014B,High,Frequent,Consistent,Discussion,Moderate,High,Late,Low Inactivity,Pass
9,24391,GGG,2013J,High,Frequent,Consistent,Learning Content,Diverse,High,Early,Low Inactivity,Distinction


In [90]:
print("Dataset shape:", student_behaviour.shape)

for feature in BEHAVIOUR_FEATURES:
    print(f"\n--- {feature} ---")
    print(
        student_behaviour[feature]
        .value_counts(dropna=False)
    )

Dataset shape: (29228, 28)

--- Activity_Level ---
Activity_Level
High      9749
Medium    9743
Low       9736
Name: count, dtype: int64

--- Learning_Frequency ---
Learning_Frequency
Moderate    9765
Frequent    9746
Rare        9717
Name: count, dtype: int64

--- Learning_Consistency ---
Learning_Consistency
Consistent            15026
Moderately Regular     8050
Irregular              6152
Name: count, dtype: int64

--- Content_Preference ---
Content_Preference
Learning Content    16931
Discussion           5609
Assessment           4384
Navigation           2304
Name: count, dtype: int64

--- Resource_Diversity ---
Resource_Diversity
Diverse     10437
Moderate     9920
Narrow       8871
Name: count, dtype: int64

--- Assessment_Engagement ---
Assessment_Engagement
High      13493
Low       11093
Medium     4642
Name: count, dtype: int64

--- Submission_Behaviour ---
Submission_Behaviour
On-Time               10208
Late                   8486
Early                  7099
No Submissio

In [91]:
print([
    col for col in activity_features.columns
    if "Content_Preference" in col
])

['Content_Preference']


In [92]:
print("Dataset shape:", student_behaviour.shape)

print("\nMissing values:")
print(
    student_behaviour[
        BEHAVIOUR_FEATURES
    ].isnull().sum()
)

print("\nFinal feature distributions:")

for feature in BEHAVIOUR_FEATURES:
    print(f"\n--- {feature} ---")

    counts = (
        student_behaviour[feature]
        .value_counts(dropna=False)
    )

    percentages = (
        student_behaviour[feature]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .mul(100)
        .round(2)
    )

    result = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages
    })

    print(result)

Dataset shape: (29228, 28)

Missing values:
Activity_Level           0
Learning_Frequency       0
Learning_Consistency     0
Content_Preference       0
Resource_Diversity       0
Assessment_Engagement    0
Submission_Behaviour     0
Inactivity_Pattern       0
dtype: int64

Final feature distributions:

--- Activity_Level ---
                Count  Percentage
Activity_Level                   
High             9749       33.36
Medium           9743       33.33
Low              9736       33.31

--- Learning_Frequency ---
                    Count  Percentage
Learning_Frequency                   
Moderate             9765       33.41
Frequent             9746       33.34
Rare                 9717       33.25

--- Learning_Consistency ---
                      Count  Percentage
Learning_Consistency                   
Consistent            15026       51.41
Moderately Regular     8050       27.54
Irregular              6152       21.05

--- Content_Preference ---
                    Count  

In [93]:
student_behaviour.to_csv(
    PROCESSED_DIR / "student_behaviour.csv",
    index=False
)

print(
    "Saved:",
    PROCESSED_DIR / "student_behaviour.csv"
)

Saved: c:\Users\Abhijith Shetty\Downloads\E-Learning-Behavior-Segmentation\data\processed\student_behaviour.csv


In [94]:
print(
    "File exists:",
    (PROCESSED_DIR / "student_behaviour.csv").exists()
)

print(
    "Final shape:",
    student_behaviour.shape
)

File exists: True
Final shape: (29228, 28)
